In [1]:
# imports
import os
import numpy as np
import pandas as pd
import neurokit2 as nk
from tqdm import tqdm

In [2]:
# Load data
PROCESSED_PATH = "../data/processed/wesad"
subjects = sorted([f.split("_")[0] for f in os.listdir(PROCESSED_PATH)])
subjects


['S10',
 'S11',
 'S13',
 'S14',
 'S15',
 'S16',
 'S17',
 'S2',
 'S3',
 'S4',
 'S5',
 'S6',
 'S7',
 'S8',
 'S9']

In [3]:
# HRV Window Parameters
WINDOW_SEC = 60
OVERLAP = 0.5

STEP_SEC = WINDOW_SEC * (1 - OVERLAP)

In [4]:
def extract_hrv_windows(rr, rr_labels, fs_rr, window_sec, step_sec):
    features = []

    rr_times = np.cumsum(rr)  # time axis in seconds
    total_time = rr_times[-1]

    start_time = 0.0
    while start_time + window_sec <= total_time:
        end_time = start_time + window_sec

        idx = np.where((rr_times >= start_time) & (rr_times < end_time))[0]

        if len(idx) < 10:
            start_time += step_sec
            continue

        rr_win = rr[idx] * 1000  # ms
        label_win = rr_labels[idx]

        # Only keep pure windows
        if not np.all(label_win == label_win[0]):
            start_time += step_sec
            continue

        label = label_win[0]

        # Convert RR intervals (ms) to peak indices
        peaks = nk.intervals_to_peaks(rr_win, sampling_rate=1000)

        hrv = nk.hrv(peaks, sampling_rate=1000, show=False)


        hrv["label"] = label
        hrv["start_time"] = start_time

        features.append(hrv)

        start_time += step_sec

    if len(features) == 0:
        return None

    return pd.concat(features, ignore_index=True)

In [5]:
# Extract HRV for all subjects
all_features = []

for subject in tqdm(subjects):
    data = np.load(os.path.join(PROCESSED_PATH, f"{subject}_rr.npz"))

    rr = data["rr_intervals"]
    rr_labels = data["rr_labels"]
    fs = data["fs"]

    df_hrv = extract_hrv_windows(
        rr,
        rr_labels,
        fs_rr=fs,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC
    )

    if df_hrv is not None:
        df_hrv["subject"] = subject
        all_features.append(df_hrv)

  0%|          | 0/15 [00:00<?, ?it/s]/home/saber/GitHub/HRV_stress_sleep/venv/lib/python3.12/site-packages/neurokit2/hrv/hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(
/home/saber/GitHub/HRV_stress_sleep/venv/lib/python3.12/site-packages/neurokit2/hrv/hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(
/home/saber/GitHub/HRV_stress_sleep/venv/lib/python3.12/site-packages/neurokit2/hrv/hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the 

In [6]:
df = pd.concat(all_features, ignore_index=True)
df.shape

(865, 94)

In [8]:
df.columns.tolist()

['HRV_MeanNN',
 'HRV_SDNN',
 'HRV_SDANN1',
 'HRV_SDNNI1',
 'HRV_SDANN2',
 'HRV_SDNNI2',
 'HRV_SDANN5',
 'HRV_SDNNI5',
 'HRV_RMSSD',
 'HRV_SDSD',
 'HRV_CVNN',
 'HRV_CVSD',
 'HRV_MedianNN',
 'HRV_MadNN',
 'HRV_MCVNN',
 'HRV_IQRNN',
 'HRV_SDRMSSD',
 'HRV_Prc20NN',
 'HRV_Prc80NN',
 'HRV_pNN50',
 'HRV_pNN20',
 'HRV_MinNN',
 'HRV_MaxNN',
 'HRV_HTI',
 'HRV_TINN',
 'HRV_ULF',
 'HRV_VLF',
 'HRV_LF',
 'HRV_HF',
 'HRV_VHF',
 'HRV_TP',
 'HRV_LFHF',
 'HRV_LFn',
 'HRV_HFn',
 'HRV_LnHF',
 'HRV_SD1',
 'HRV_SD2',
 'HRV_SD1SD2',
 'HRV_S',
 'HRV_CSI',
 'HRV_CVI',
 'HRV_CSI_Modified',
 'HRV_PIP',
 'HRV_IALS',
 'HRV_PSS',
 'HRV_PAS',
 'HRV_GI',
 'HRV_SI',
 'HRV_AI',
 'HRV_PI',
 'HRV_C1d',
 'HRV_C1a',
 'HRV_SD1d',
 'HRV_SD1a',
 'HRV_C2d',
 'HRV_C2a',
 'HRV_SD2d',
 'HRV_SD2a',
 'HRV_Cd',
 'HRV_Ca',
 'HRV_SDNNd',
 'HRV_SDNNa',
 'HRV_DFA_alpha1',
 'HRV_MFDFA_alpha1_Width',
 'HRV_MFDFA_alpha1_Peak',
 'HRV_MFDFA_alpha1_Mean',
 'HRV_MFDFA_alpha1_Max',
 'HRV_MFDFA_alpha1_Delta',
 'HRV_MFDFA_alpha1_Asymmetry',
 'HR

# Only keeping essential columns for baseline model

In [12]:
FEATURE_COLS = [
    "HRV_MeanNN",
    "HRV_SDNN",
    "HRV_RMSSD",
    "HRV_pNN50",
    "HRV_LF",
    "HRV_HF",
    "HRV_LFHF"
]


In [13]:
sorted([c for c in df.columns if c.startswith("HRV_")])

['HRV_AI',
 'HRV_ApEn',
 'HRV_C1a',
 'HRV_C1d',
 'HRV_C2a',
 'HRV_C2d',
 'HRV_CD',
 'HRV_CMSEn',
 'HRV_CSI',
 'HRV_CSI_Modified',
 'HRV_CVI',
 'HRV_CVNN',
 'HRV_CVSD',
 'HRV_Ca',
 'HRV_Cd',
 'HRV_DFA_alpha1',
 'HRV_DFA_alpha2',
 'HRV_FuzzyEn',
 'HRV_GI',
 'HRV_HF',
 'HRV_HFD',
 'HRV_HFn',
 'HRV_HTI',
 'HRV_IALS',
 'HRV_IQRNN',
 'HRV_KFD',
 'HRV_LF',
 'HRV_LFHF',
 'HRV_LFn',
 'HRV_LZC',
 'HRV_LnHF',
 'HRV_MCVNN',
 'HRV_MFDFA_alpha1_Asymmetry',
 'HRV_MFDFA_alpha1_Delta',
 'HRV_MFDFA_alpha1_Fluctuation',
 'HRV_MFDFA_alpha1_Increment',
 'HRV_MFDFA_alpha1_Max',
 'HRV_MFDFA_alpha1_Mean',
 'HRV_MFDFA_alpha1_Peak',
 'HRV_MFDFA_alpha1_Width',
 'HRV_MFDFA_alpha2_Asymmetry',
 'HRV_MFDFA_alpha2_Delta',
 'HRV_MFDFA_alpha2_Fluctuation',
 'HRV_MFDFA_alpha2_Increment',
 'HRV_MFDFA_alpha2_Max',
 'HRV_MFDFA_alpha2_Mean',
 'HRV_MFDFA_alpha2_Peak',
 'HRV_MFDFA_alpha2_Width',
 'HRV_MSEn',
 'HRV_MadNN',
 'HRV_MaxNN',
 'HRV_MeanNN',
 'HRV_MedianNN',
 'HRV_MinNN',
 'HRV_PAS',
 'HRV_PI',
 'HRV_PIP',
 'HRV_PSS'

In [14]:
df.groupby("label")[FEATURE_COLS].mean()

,HRV_MeanNN,HRV_SDNN,HRV_RMSSD,HRV_pNN50,HRV_LF,HRV_HF,HRV_LFHF
label,,,,,,,
1,847.096454,69.490228,52.380655,26.465879,0.031539,0.030587,2.047687
2,639.586359,63.601458,45.210852,13.651297,0.034570,0.027142,2.846588


In [15]:
# Saving the data
df.to_csv("../data/processed/wesad_hrv_features.csv", index=False)